# Notebook 2: benchmarking decision models

How good are decision models at walking a knowledge graph, and what do they cost? This notebook answers that with three
well-known question-answering benchmarks:

1. our results for several decision models and an LLM (GPT-5.6 Luna),
2. your turn: run 50 questions with a model of your choice,
3. compare your run with ours on the same questions.

A run of 50 questions takes a few minutes. It costs about $0.01 to $0.03 with a decision model, and about $0.07 with
GPT-5.6 Luna. The `benchmark/` folder has the full version of these steps, for any number of questions and models.

## The three benchmarks

| Benchmark | Questions | Graph | Score |
|-|-|-|-|
| MetaQA | Questions about films, 1 to 3 steps | The film graph from notebook 1 | Hits@1: the chance that a random pick from the answer is right |
| WebQSP | Real questions that people typed into a search engine | A small Freebase graph for each question | F1: how well the answer set matches the right answers |
| ComplexWebQuestions (CWQ) | Harder questions built from WebQSP | A small Freebase graph for each question | P@1: like Hits@1, and an answer may match any known name of a right answer |

All scores run from 0 to 1, and higher is better. "Exactly correct" counts the questions whose answer was fully right.

A walk ends on a set of entities, and each benchmark scores that set in its own way. Here are three real walks by
d1-ultra, one from each benchmark:

![How each benchmark scores an answer: Hits@1 for MetaQA, F1 for WebQSP and P@1 for CWQ, each with a worked example](images/benchmark_scoring.svg)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

from utils import DecisionModel, LLM, display, graph, load_results, open_run, questions_for, run_models
from utils.settings import PUBLISHED

## Our results

We ran six decision models and GPT-5.6 Luna on the full benchmarks. The results, with charts and costs, are in
[`benchmark/RESULTS.md`](../benchmark/RESULTS.md). In short, the best decision models were as accurate as GPT-5.6 Luna,
or more accurate, and GPT-5.6 Luna cost 6 to 12 times as much.

If per-question results are added to `benchmark/published_results/`, the cell below shows them here too.

In [ ]:
published = load_results(root=PUBLISHED)
ours = None
if published.empty:
    print("No per-question results in benchmark/published_results/ yet. See benchmark/RESULTS.md.")
else:
    ours = display.summary(published)
    display.plot_score_cost(ours)
ours

## Your turn: run 50 questions

A run walks every question, scores every answer, and adds up the scores and costs. Here is what happens to one
question in a run of d1-ultra on 50 WebQSP questions:

![One benchmark run: take the questions, load their graphs, walk each question, score the answer, and add up all the questions](images/benchmark_run.svg)

Pick a benchmark and a model. Decision models include `liquid/d1-ultra`, `inception/mercury-decide`,
`typesafe/jev-1.14` and `microsoft/microsoft-decision-1`. Any LLM on OpenRouter that supports structured output works
too, for example `LLM("openai/gpt-5.6-luna")`. The model names are on https://openrouter.ai/models.

In [ ]:
BENCHMARK = "webqsp"   # "metaqa", "webqsp" or "cwq"
MODEL = DecisionModel("liquid/d1-ultra")
QUESTIONS = 50

In [ ]:
questions = questions_for(BENCHMARK, size=QUESTIONS)
driver = graph.connect()
info = graph.ensure_loaded(driver, BENCHMARK, questions)
driver.close()
print(f"{len(questions)} questions ready; their graphs hold {info['entities']:,} entities and {info['facts']:,} facts.")

run = open_run(BENCHMARK, [{k: v for k, v in q.items() if k != "graph"} for q in questions],
               name=f"my-{BENCHMARK}-{QUESTIONS}")
await run_models(run, [MODEL], budget_usd=0.50)
mine = load_results(BENCHMARK, [run.name])
display.summary(mine)

## Compare with our results on the same questions

The table keeps only the questions that you ran, so every model is scored on the same questions.

In [ ]:
import pandas as pd

if published.empty:
    print("Our results are not added yet, so here are yours on their own.")
    table = display.summary(mine)
else:
    same = published[(published["benchmark"] == BENCHMARK) & published["qid"].isin(set(mine["qid"]))]
    table = display.summary(pd.concat([same, mine]))
display.plot_score_cost(table)
table

## Look at one walk

Here is a question that your model got wrong, with its decision log. Compare the right answers with the path it
took.

In [ ]:
from utils import load_decisions

wrong = mine[mine["exact"] == False]  # noqa: E712
log = None
if wrong.empty:
    print("Every answer was exactly right. Try another benchmark or model!")
else:
    row = wrong.iloc[0]
    gold = next(q for q in questions if q["qid"] == row["qid"])["answers"]
    print("Question:     ", row["question"])
    print("Right answers:", ", ".join(gold[:8]))
    print("Path:         ", " -> ".join(row["steps"]) or "(stopped at once)")
    print("Walk's answer:", ", ".join((row["final"] or [])[:8]))
    log = display.walk_log(load_decisions(run.name, BENCHMARK, MODEL.label)[row["qid"]])
log

## What next

- To run more questions, more models, or all three benchmarks, use the notebooks in `benchmark/`.
- Notebook 3 answers questions on a Neo4j graph with Neo4j's GraphRAG library, with a decision model walking the
  graph.